# 03 — Evaluate Base vs Fine-tuned

Runs both the base Qwen 2.5-1.5B-Instruct and the LoRA-adapted version against
the held-out test set, then plots the comparison.

**Runtime:** ~45-90 minutes on a free T4 (300 test examples × 2 models, most
time is generation, not the metric math).

**Recovery:** Evaluation is NOT resumable mid-run — if Colab disconnects partway
through, you'll need to restart from scratch. To minimize risk:
1. Run the smoke test cell first (~3 min) to validate end-to-end before
   committing to the full ~1 hour eval.
2. Only run the full eval when you have ~1.5 hours of GPU quota left.

In [ ]:
!git clone https://github.com/masonsau0/logistics-qa-lora.git
%cd logistics-qa-lora
!pip install -q -r requirements.txt

In [ ]:
# Restore dataset + adapter from Drive
from google.colab import drive

drive.mount("/content/drive")
!cp /content/drive/MyDrive/logistics-qa-lora/data/*.jsonl data/
!mkdir -p artifacts/checkpoints
!cp -r /content/drive/MyDrive/logistics-qa-lora/artifacts/final artifacts/checkpoints/
!ls artifacts/checkpoints/final/

## Smoke test (recommended — ~3-5 minutes)

Evaluates both models on just 10 test examples instead of 300. Confirms the
adapter loads, generation works on T4, and the metrics pipeline runs end-to-end.

If this finishes cleanly with a summary table, the full eval will work too.

In [ ]:
# Smoke test — eval on 10 examples instead of 300 (~3-5 min on T4)
# Confirms: adapter loads, base + fine-tuned both generate, metrics compute, JSON writes.
# Output goes to a separate file so it doesn't overwrite the real eval results.
!python -m src.evaluate --compare --limit 10 --output artifacts/results/eval_smoke.json

## Run the evaluation

In [ ]:
!python -m src.evaluate --compare --output artifacts/results/eval.json

## Plot the per-category comparison

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

results = json.load(open("artifacts/results/eval.json"))
base = results["base"]["per_category"]
ft = results["fine_tuned"]["per_category"]

cats = sorted(base.keys())
base_em = [base[c]["em"] for c in cats]
ft_em = [ft[c]["em"] for c in cats]

x = np.arange(len(cats))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.2, base_em, 0.4, label="Base")
ax.bar(x + 0.2, ft_em, 0.4, label="+ LoRA")
ax.set_xticks(x)
ax.set_xticklabels([c.replace("_", "\n") for c in cats], fontsize=9)
ax.set_ylabel("Exact Match")
ax.set_title("Per-category EM: Qwen 2.5-1.5B base vs LoRA fine-tuned")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("artifacts/results/per_category_em.png", dpi=150)
plt.show()

print(
    f"\nOverall: base EM = {results['base']['overall']['em']:.3f}, fine-tuned EM = {results['fine_tuned']['overall']['em']:.3f}"
)
print(f"Δ EM = {results['delta']['em']:+.3f}")
print(f"Δ F1 = {results['delta']['f1']:+.3f}")

## Inspect some specific examples

In [ ]:
# Find examples where the fine-tune helped the most
base_preds = {p["question"]: p for p in results["base"]["predictions"]}
ft_preds = {p["question"]: p for p in results["fine_tuned"]["predictions"]}

improvements = []
for q, ft_p in ft_preds.items():
    if q in base_preds:
        delta = ft_p["em"] - base_preds[q]["em"]
        improvements.append((delta, q, base_preds[q], ft_p))

improvements.sort(reverse=True)

for delta, q, b, f in improvements[:3]:
    print("=" * 70)
    print(f"Δ EM = {delta:+.2f}   [{b['category']}]")
    print(f"Q: {q}")
    print(f"\nBase   ({b['em']:.2f}): {b['prediction'][:400]}")
    print(f"\nLoRA   ({f['em']:.2f}): {f['prediction'][:400]}")
    print(f"\nRef: {b['reference'][:400]}")
    print()

## Save the results

Copy `artifacts/results/eval.json` and `per_category_em.png` to your repo and paste the summary table into the README.